In [48]:
import pandas as pd

nhanes = pd.read_csv("../datasets/nhanes.csv")

## Preprocessing Pipeline

In [49]:
#Processing for targets


def bpSYmap(bp):
    if bp >= 120:
        return 1
    else:
        return 0
def bpDImap(bp):
    if bp >= 80:
        return 1
    else:
        return 0 

def normmap(bp):
    if bp == 0:
        return "normal"
    else:
        return "abnormal"

def cholmap(chol):
    if chol < 200:
        return "normal"
    elif chol <= 239:
        return "borderline"
    else:
        return "high"


#sum the two maps; now we have nonzero values where systolic or diasystolic is high
nhanes["BP"] = nhanes["BPXOSY1"].map(bpSYmap) + nhanes["BPXODI1"].map(bpDImap)

#apply the text labels
nhanes["BP"] = nhanes["BP"].map(normmap)


nhanes["CHOL"] = nhanes["LBXTC"].map(cholmap)
nhanes.drop(labels=["BPXOSY1", "BPXODI1", "LBXTC"], axis=1, inplace=True)

In [50]:
nhanes

,SEQN,SDDSRVYR,RIDSTATR,RIAGENDR,RIDAGEYR,RIDRETH1,DMQMILIZ,DMDBORN4,DMDYRUSR,DMDEDUC2,...,HIQ032D,HIQ032F,HIQ032H,HIQ032I,HIQ210,INDFMMPC,INQ300,IND310,BP,CHOL
0,132141.0,12.0,2.0,2.0,56.0,4.0,2.0,1.0,NaN,5.0,...,NaN,NaN,NaN,NaN,2.0,3.0,1.0,NaN,normal,borderline
1,136977.0,12.0,2.0,1.0,29.0,3.0,2.0,2.0,2.0,5.0,...,4.0,NaN,NaN,NaN,2.0,3.0,2.0,2.0,abnormal,normal
2,134280.0,12.0,2.0,2.0,25.0,3.0,2.0,1.0,NaN,5.0,...,NaN,NaN,NaN,NaN,2.0,3.0,1.0,NaN,normal,normal
3,132522.0,12.0,2.0,2.0,64.0,3.0,2.0,1.0,NaN,4.0,...,NaN,NaN,NaN,NaN,2.0,NaN,NaN,NaN,abnormal,borderline
4,138725.0,12.0,2.0,2.0,28.0,3.0,2.0,1.0,NaN,5.0,...,NaN,NaN,NaN,NaN,2.0,3.0,1.0,NaN,normal,borderline
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5698,138255.0,12.0,2.0,2.0,65.0,5.0,2.0,1.0,NaN,4.0,...,NaN,NaN,8.0,NaN,2.0,1.0,2.0,1.0,normal,normal
5699,134584.0,12.0,2.0,1.0,56.0,1.0,2.0,1.0,NaN,4.0,...,NaN,NaN,NaN,NaN,2.0,3.0,2.0,3.0,abnormal,borderline
5700,133659.0,12.0,2.0,2.0,80.0,3.0,2.0,1.0,NaN,4.0,...,NaN,NaN,NaN,9.0,2.0,NaN,NaN,NaN,abnormal,normal
5701,131701.0,12.0,2.0,2.0,46.0,3.0,2.0,1.0,NaN,3.0,...,NaN,NaN,NaN,NaN,1.0,2.0,1.0,NaN,abnormal,high


In [51]:
bp_Rock_Doves = nhanes.copy()
chol_Rock_Doves = nhanes.copy()

from sklearn.model_selection import train_test_split

bp_training_Rock_Doves, bp_test_Rock_Doves = train_test_split(bp_Rock_Doves, train_size=.8, random_state=42, stratify=bp_Rock_Doves["BP"])

chol_training_Rock_Doves, chol_test_Rock_Doves = train_test_split(chol_Rock_Doves, train_size=.8, random_state=42, stratify=chol_Rock_Doves["CHOL"])

In [52]:
#It's possible that this can be reduced to fewer pipelines using metadata routing or more column transformers

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import FunctionTransformer
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler



#We need to remove placehold values before imputing

#drop SEQN, SDDSRVYR, RIDSTATR



#remove 7, 9: INQ300 INDFMMPC HIQ210 HIQ011 ALQ151 ALQ111 DMDEDUC2 DMQMILIZ


#Pipeline to remove 7/9 placeholder values
def singleremover(dat):
    if dat == 7 or dat == 9:
        return np.nan
    else:
        return dat
    
def singleremovermap(df):
    for column in df.columns:
        df[column] = df[column].map(singleremover)
    return df

srtransform = FunctionTransformer(singleremovermap)

#Remove placeholders for ordinal values that do not need to be 1-hotted
srpipe = Pipeline([
    ("remove 7/9", srtransform),
    ("impute", SimpleImputer()),
    ("scale", MinMaxScaler())
])

#Pipeline including 1-hot encoder for categorical 
srhotpipe = Pipeline([
    ("remove 7/9", srtransform),
    ("impute", SimpleImputer(strategy="most_frequent")),     #don't want to use mean if we're going to 1hot
    ("onehot", OneHotEncoder(min_frequency=0.01))

])

#remove 77, 99: DMDBORN4, IND310, HIQ032A, ALQ121, DMDMARTZ, DMDYRUSR

def doubleeremover(dat):
    if dat == 77 or dat == 99:
        return np.nan
    else:
        return dat
    
def doubleremovermap(df):
    for column in df.columns:
        df[column] = df[column].map(doubleeremover)
    return df

drtransform = FunctionTransformer(doubleremovermap)

drpipe = Pipeline([
    ("remove 77/99", drtransform),
    ("impute", SimpleImputer()),
    ("scale", MinMaxScaler())
])


drhotpipe = Pipeline([
    ("remove 77/99", drtransform),
    ("impute", SimpleImputer(strategy="most_frequent")),     #don't want to use mean if we're going to 1hot
    ("onehot", OneHotEncoder(min_frequency=0.01))
])


#remove 777, 999: ALQ130, ALQ170

def tripleeremover(dat):
    if dat == 777 or dat == 999:
        return np.nan
    else:
        return dat
    
def tripleremovermap(df):
    for column in df.columns:
        df[column] = df[column].map(tripleeremover)
    return df

trtransform = FunctionTransformer(tripleremovermap)

trpipe = Pipeline([
    ("remove 777/999", trtransform),
    ("impute", SimpleImputer()),
    ("scale", MinMaxScaler())
])

trhotpipe = Pipeline([
    ("remove 777/999", trtransform),
    ("impute", SimpleImputer(strategy="most_frequent")),     #don't want to use mean if we're going to 1hot
    ("onehot", OneHotEncoder(min_frequency=0.01))
])


hotpipe = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(min_frequency=0.01))
])



#special pipeline transform for ALQ121, which has values out of order; "never" is ranked highest

def alc(dat):
    if dat == 77 or dat == 99:
        return np.nan
    if dat == 0:
        return 11       #to put the "never" value in its proper place
    else:
        return dat
    
def alcmap(df):
    for column in df.columns:
        df[column] = df[column].map(alc)
    return df

alctransform = FunctionTransformer(alcmap)

alcpipe = Pipeline([
    ("remove 777/999", alctransform),
    ("impute", SimpleImputer()),
    ("scale", MinMaxScaler())
])


#Special transform for RIDAGEYR, which is capped at 80 but has an average value for the capped bucket of 85

def age(dat):
    if dat == 80:
        return 85
    else:
        return dat
    
def agemap(df):
    for column in df.columns:
        df[column] = df[column].map(age)
    return df

agetransform = FunctionTransformer(agemap)

agepipe = Pipeline([
    ("remove 777/999", agetransform),
    ("impute", SimpleImputer()),
    ("scale", MinMaxScaler())
])



#pipeline to impute for features that need no other proccessing
imppipe = Pipeline([
    ("impute", SimpleImputer())         #can't remove the 7/9 values here because the values to be removed are different for different columns
])

In [53]:
plineb = ColumnTransformer([
    #drop seqn and SDDSRVYR, RIDSTATR


    ("remove 7/9", srpipe, ["INDFMMPC",  "HIQ011", "ALQ151", "ALQ111", "DMDEDUC2"]),
    ("remove 7/9 and 1hot", srhotpipe, ["DMQMILIZ", "HIQ210", "INQ300"]),
    
    ("remove 77/99", drpipe, ["IND310", "HIQ032A", "ALQ121",  "DMDYRUSR"]),
    ("remove 77/99 and 1hot", drhotpipe, ["DMDBORN4", "DMDMARTZ"]),
    
    ("remove 777/999", trpipe, ["ALQ130", "ALQ170"]),


    ("remove 77/99 and set 0 = 11", alcpipe, ["ALQ121"]),

    #impute all the rest. All transforms are done concurrently so had to impute the above ones seperatesly

    ("impute", imppipe, [ 
       'DMDHHSIZ',
        'ALQ151',
       'HIQ011',  
        #'HIQ032B', 'HIQ032D', 'HIQ032F', 'HIQ032H', 'HIQ032I',
        'INDFMMPC']),

    ("fix age bucket and impute", agepipe, [ 'RIDAGEYR']),
    
    ("onehot", hotpipe, ["RIAGENDR", "RIDRETH1"])



], remainder="drop")    #everything is at least going through the imputer. Would be faster to not imput when we know there are no NaNs?

In [54]:
#unless we decide to do different preprocessing on chol and bp
plinec=plineb

## Model Training

In [55]:
from sklearn.model_selection import RandomizedSearchCV

#perform random search and return best model. Pass X and y so the search isn't hard-coded to one target.
def RandSearch(model, params, X, y, scoring="f1_macro", n_iter=20, cv=10, random_state=42):
    RS = RandomizedSearchCV(
        model, params, n_iter=n_iter, cv=cv, scoring=scoring,
        random_state=random_state, n_jobs=-1, refit=True
    ).fit(X=X, y=y)
    print("best score:", RS.best_score_, "best params:", RS.best_params_)
    return RS

In [56]:
#BP Models

from sklearn.svm import SVC

from sklearn.ensemble import AdaBoostClassifier

from sklearn.linear_model import SGDClassifier




Svc = Pipeline([
    ("preprocces", plineb),             #assuming for simpliciy that we use the same pipeline for bp and chol
    ("SVC", SVC(random_state=42))
])

AdaBC = Pipeline([
    ("preprocces", plineb),             #assuming for simpliciy that we use the same pipeline for bp and chol
    ("AdaBoost", AdaBoostClassifier(random_state=42))
])

SgdC = Pipeline([
    ("preprocces", plineb),             #assuming for simpliciy that we use the same pipeline for bp and chol
    ("SGDClassifier", SGDClassifier(random_state=42))
])

In [57]:
#chol models

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

#DecisionTreeClassifier, KNeighborsClassifier, and RandomForestClassifier

Dtc = Pipeline([
    ("preprocces", plineb),             #assuming for simpliciy that we use the same pipeline for bp and chol
    ("DTC", DecisionTreeClassifier(random_state=42))
])

Knn = Pipeline([
    ("preprocces", plineb),             #assuming for simpliciy that we use the same pipeline for bp and chol
    ("KNN", KNeighborsClassifier())
])

Rfc = Pipeline([
    ("preprocces", plineb),             #assuming for simpliciy that we use the same pipeline for bp and chol
    ("RandomForest", RandomForestClassifier(random_state=42))
])

In [58]:
#example using just a few hyperparameters. This will be very large but if we have a lot of compute that's fine. Practically we will want to choose fewer
#NOTE: SGDClassifier is a BP finalist; pass the BP training split + BP labels.
bestSGD = RandSearch(
    SgdC,
    dict(
        SGDClassifier__loss=['hinge', 'log_loss', 'modified_huber', 'squared_hinge', 'perceptron'],
        SGDClassifier__alpha=[0.0005, 0.0001, 0.00015],
        SGDClassifier__penalty=['l2', 'l1', 'elasticnet', None],
    ),
    X=bp_training_Rock_Doves.drop(columns=["BP", "CHOL"]),
    y=np.ravel(bp_training_Rock_Doves[["BP"]]),
)

/home/selinajcheng/College/CSCI353_ML/Project/.venv/lib/python3.12/site-packages/sklearn/linear_model/_stochastic_gradient.py:733: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(
/home/selinajcheng/College/CSCI353_ML/Project/.venv/lib/python3.12/site-packages/sklearn/linear_model/_stochastic_gradient.py:733: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(
/home/selinajcheng/College/CSCI353_ML/Project/.venv/lib/python3.12/site-packages/sklearn/linear_model/_stochastic_gradient.py:733: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(
/home/selinajcheng/College/CSCI353_ML/Project/.venv/lib/python3.12/site-packages/sklearn/linear_model/_stochastic_gradient.py:733: ConvergenceWarning: Maximum number of iteration re

best score: 0.6366719105772985 best params: {'SGDClassifier__penalty': 'l1', 'SGDClassifier__loss': 'log_loss', 'SGDClassifier__alpha': 0.0005}


## Model Evaluation

In [59]:
#Dummy model for testing

from sklearn.dummy import DummyClassifier


In [60]:
from sklearn.model_selection import cross_validate

#cross validate model on both datasets and return results as bp, chol
def CV(model):
    print(list(model.named_steps.keys())[-1])
    bp = cross_validate(estimator=model, X=bp_training_Rock_Doves.drop(columns=["BP", "CHOL"]), y=np.ravel(bp_training_Rock_Doves[["BP"]]),  cv=10, scoring=["f1_macro", "recall_macro", "accuracy"])

    print("BP",
            bp)

    chol = cross_validate(estimator=model, X=chol_training_Rock_Doves.drop(columns=["BP", "CHOL"]), y=np.ravel(chol_training_Rock_Doves[["CHOL"]]),  cv=10, scoring=["f1_macro", "recall_macro", "accuracy"])

    print("CHOL",
        chol)
    return bp, chol

In [61]:
Svc

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocces', ...), ('SVC', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('remove 7/9', ...), ('remove 7/9 and 1hot', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the d

In [62]:
CVresults = {}

for model in [Svc, AdaBC, SgdC, Dtc, Knn, Rfc]:
    CVresults[model.steps[-1][0]] = CV(model)

SVC
BP {'fit_time': array([1.05538774, 0.72431493, 0.79508471, 0.77359176, 0.85805178,
       0.84262276, 0.76587152, 0.79624653, 0.81710339, 0.79907894]), 'score_time': array([0.13416076, 0.1440897 , 0.12844086, 0.13418627, 0.15633702,
       0.13110995, 0.13781714, 0.13501811, 0.13915157, 0.13211656]), 'test_f1_macro': array([0.62050173, 0.67461893, 0.62524351, 0.6375642 , 0.61830357,
       0.62575915, 0.6464346 , 0.63542644, 0.6340158 , 0.64287604]), 'test_recall_macro': array([0.62196135, 0.67528502, 0.62858252, 0.64100971, 0.62516505,
       0.62670974, 0.64917642, 0.63596415, 0.63869973, 0.64952564]), 'test_accuracy': array([0.6345733 , 0.6892779 , 0.64473684, 0.65789474, 0.64473684,
       0.63815789, 0.66447368, 0.64692982, 0.65570175, 0.66885965])}
CHOL {'fit_time': array([1.24022961, 1.40438533, 0.63768363, 1.3913815 , 1.37113476,
       1.22550035, 1.21398687, 1.23493767, 1.26702571, 1.28426075]), 'score_time': array([0.1356566 , 0.13654804, 0.14076209, 0.13619542, 0.134074

In [63]:
CVresults

{'SVC': ({'fit_time': array([1.05538774, 0.72431493, 0.79508471, 0.77359176, 0.85805178,
          0.84262276, 0.76587152, 0.79624653, 0.81710339, 0.79907894]),
   'score_time': array([0.13416076, 0.1440897 , 0.12844086, 0.13418627, 0.15633702,
          0.13110995, 0.13781714, 0.13501811, 0.13915157, 0.13211656]),
   'test_f1_macro': array([0.62050173, 0.67461893, 0.62524351, 0.6375642 , 0.61830357,
          0.62575915, 0.6464346 , 0.63542644, 0.6340158 , 0.64287604]),
   'test_recall_macro': array([0.62196135, 0.67528502, 0.62858252, 0.64100971, 0.62516505,
          0.62670974, 0.64917642, 0.63596415, 0.63869973, 0.64952564]),
   'test_accuracy': array([0.6345733 , 0.6892779 , 0.64473684, 0.65789474, 0.64473684,
          0.63815789, 0.66447368, 0.64692982, 0.65570175, 0.66885965])},
  {'fit_time': array([1.24022961, 1.40438533, 0.63768363, 1.3913815 , 1.37113476,
          1.22550035, 1.21398687, 1.23493767, 1.26702571, 1.28426075]),
   'score_time': array([0.1356566 , 0.13654804,

In [64]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint
from scipy.stats import uniform

## Hyperparameter tuning and preprocessor refinement

Also revised `plinec` to drop `INDFMMPC` and `INQ300` and remove duplication of `INDFMMPC`, `ALQ151`, and `HIQ011` across the `imp_passthrough` tuple for the column transformer. BP continues using `plineb` since changing the BP preprocessor didn't improve held-out scores.

In [65]:
# Reassign plinec to drop INDFMMPC and INQ300 (since Step 3 plan flagged as redundant) and remove the duplication of INDFMMPC, ALQ151, and HIQ011 that previously appeared.
# BP continues to use plineb (above). Only CHOL uses this reassigned plinec.
plinec = ColumnTransformer([
    ("remove 7/9", srpipe, ["HIQ011", "ALQ151", "ALQ111", "DMDEDUC2"]),
    ("remove 7/9 and 1hot", srhotpipe, ["DMQMILIZ", "HIQ210"]),
    ("remove 77/99", drpipe, ["IND310", "HIQ032A", "ALQ121", "DMDYRUSR"]),
    ("remove 77/99 and 1hot", drhotpipe, ["DMDBORN4", "DMDMARTZ"]),
    ("remove 777/999", trpipe, ["ALQ130", "ALQ170"]),
    ("remove 77/99 and set 0 = 11", alcpipe, ["ALQ121"]),
    ("impute", imppipe, ["DMDHHSIZ"]),
    ("fix age bucket and impute", agepipe, ["RIDAGEYR"]),
    ("onehot", hotpipe, ["RIAGENDR", "RIDRETH1"]),
], remainder="drop")

In [66]:
# Tuned AdaBoost for BP. RandomizedSearchCV picked learning_rate=0.236 and n_estimators=239 (default base estimator: stump).
# Uses plineb (original preprocessor). Among the three BP finalists, AdaBoost / SVC / SGD were statistically tied on every metric (ANOVA below). AdaBoost was picked by highest mean macro-F1 across folds
bp_tuned = Pipeline([
    ("preprocces", plineb),
    ("AdaBoost", AdaBoostClassifier(
        random_state=42,
        learning_rate=0.2359096517992312,
        n_estimators=239,
    )),
])
bp_tuned.fit(
    bp_training_Rock_Doves.drop(columns=["BP", "CHOL"]),
    np.ravel(bp_training_Rock_Doves[["BP"]]),
)
bp_tuned

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocces', ...), ('AdaBoost', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('remove 7/9', ...), ('remove 7/9 and 1hot', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of 

In [67]:
# Tuned RandomForest for CHOL. RandomizedSearchCV with class_weight in search distribution (Step 3 plan listed class_weight as a tuning target while initial search hard-fixed it to "balanced") picked.
# Uses the reassigned plinec above
chol_tuned = Pipeline([
    ("preprocces", plinec),
    ("RandomForest", RandomForestClassifier(
        random_state=42,
        class_weight="balanced_subsample",
        n_estimators=235,
        max_depth=13,
        min_samples_leaf=12,
        max_features="sqrt",
    )),
])
chol_tuned.fit(
    chol_training_Rock_Doves.drop(columns=["BP", "CHOL"]),
    np.ravel(chol_training_Rock_Doves[["CHOL"]]),
)
chol_tuned

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocces', ...), ('RandomForest', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('remove 7/9', ...), ('remove 7/9 and 1hot', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output

### Statistical comparison across CV folds

In [68]:
from scipy.stats import f_oneway, ttest_rel
from sklearn.dummy import DummyClassifier

SCORERS = ["f1_macro", "recall_macro", "accuracy"]

bp_X = bp_training_Rock_Doves.drop(columns=["BP", "CHOL"])
bp_y = np.ravel(bp_training_Rock_Doves[["BP"]])
chol_X = chol_training_Rock_Doves.drop(columns=["BP", "CHOL"])
chol_y = np.ravel(chol_training_Rock_Doves[["CHOL"]])

bp_dummy_pipe = Pipeline([("clf", DummyClassifier(strategy="most_frequent"))])
chol_dummy_pipe = Pipeline([("clf", DummyClassifier(strategy="most_frequent"))])

print("Running cross-validation on the tuned BP/CHOL pipelines and the dummies...")
cv_bp_tuned = cross_validate(bp_tuned, bp_X, bp_y, cv=10, scoring=SCORERS)
cv_chol_tuned = cross_validate(chol_tuned, chol_X, chol_y, cv=10, scoring=SCORERS)
cv_bp_dummy = cross_validate(bp_dummy_pipe, bp_X, bp_y, cv=10, scoring=SCORERS)
cv_chol_dummy = cross_validate(chol_dummy_pipe, chol_X, chol_y, cv=10, scoring=SCORERS)
print("Done.\n")

def baseline_folds(model_step_name, target_idx, metric):
    # CVresults indexed by the last pipeline step name. target_idx 0 = BP, 1 = CHOL.
    return CVresults[model_step_name][target_idx][f"test_{metric}"]

def show(label, p):
    sig = "(SIGNIFICANT at α=0.05)" if p < 0.05 else "(NOT significant at α=0.05)"
    print(f"  {label}  p={p:.4f} {sig}")

print("=== BP finalists at default hyperparameters (3-way ANOVA): AdaBoost / SVC / SGD ===")
for m in SCORERS:
    a = baseline_folds("AdaBoost", 0, m)
    b = baseline_folds("SVC", 0, m)
    c = baseline_folds("SGDClassifier", 0, m)
    F, p = f_oneway(a, b, c)
    show(f"BP {m}: F={F:.3f}  means: AdaBoost {a.mean():.3f}, SVC {b.mean():.3f}, SGD {c.mean():.3f}", p)

print("\n=== CHOL finalists at default hyperparameters (3-way ANOVA): DTC / KNN / RandomForest ===")
for m in SCORERS:
    a = baseline_folds("DTC", 1, m)
    b = baseline_folds("KNN", 1, m)
    c = baseline_folds("RandomForest", 1, m)
    F, p = f_oneway(a, b, c)
    show(f"CHOL {m}: F={F:.3f}  means: DTC {a.mean():.3f}, KNN {b.mean():.3f}, RF {c.mean():.3f}", p)

print("\n=== Paired t-tests: tuned pick vs default-hyperparameter baseline (same algorithm family) ===")
for m in SCORERS:
    a, b = cv_bp_tuned[f"test_{m}"], baseline_folds("AdaBoost", 0, m)
    t, p = ttest_rel(a, b)
    show(f"BP tuned vs baseline AdaBoost: {m} diff={a.mean()-b.mean():+.4f}", p)
for m in SCORERS:
    a, b = cv_chol_tuned[f"test_{m}"], baseline_folds("DTC", 1, m)
    t, p = ttest_rel(a, b)
    show(f"CHOL tuned RF vs baseline DTC: {m} diff={a.mean()-b.mean():+.4f}", p)

print("\n=== ANOVA: tuned pick vs always-majority Dummy ===")
for m in SCORERS:
    a, b = cv_bp_tuned[f"test_{m}"], cv_bp_dummy[f"test_{m}"]
    F, p = f_oneway(a, b)
    show(f"BP tuned vs dummy: {m}  means: model {a.mean():.3f}, dummy {b.mean():.3f}", p)
for m in SCORERS:
    a, b = cv_chol_tuned[f"test_{m}"], cv_chol_dummy[f"test_{m}"]
    F, p = f_oneway(a, b)
    show(f"CHOL tuned vs dummy: {m}  means: model {a.mean():.3f}, dummy {b.mean():.3f}", p)

Running cross-validation on the tuned BP/CHOL pipelines and the dummies...
Done.

=== BP finalists at default hyperparameters (3-way ANOVA): AdaBoost / SVC / SGD ===
  BP f1_macro: F=8.106  means: AdaBoost 0.635, SVC 0.636, SGD 0.588  p=0.0017 (SIGNIFICANT at α=0.05)
  BP recall_macro: F=5.737  means: AdaBoost 0.636, SVC 0.639, SGD 0.607  p=0.0084 (SIGNIFICANT at α=0.05)
  BP accuracy: F=4.712  means: AdaBoost 0.649, SVC 0.655, SGD 0.617  p=0.0176 (SIGNIFICANT at α=0.05)

=== CHOL finalists at default hyperparameters (3-way ANOVA): DTC / KNN / RandomForest ===
  CHOL f1_macro: F=4.692  means: DTC 0.349, KNN 0.354, RF 0.329  p=0.0178 (SIGNIFICANT at α=0.05)
  CHOL recall_macro: F=0.338  means: DTC 0.350, KNN 0.357, RF 0.354  p=0.7165 (NOT significant at α=0.05)
  CHOL accuracy: F=97.057  means: DTC 0.434, KNN 0.474, RF 0.549  p=0.0000 (SIGNIFICANT at α=0.05)

=== Paired t-tests: tuned pick vs default-hyperparameter baseline (same algorithm family) ===
  BP tuned vs baseline AdaBoost: f1

### Held-out test set evaluation

In [69]:
from sklearn.metrics import f1_score, recall_score, accuracy_score

def eval_on_test(label, model, X_test, y_test):
    y_pred = model.predict(X_test)
    f1 = f1_score(y_test, y_pred, average="macro")
    rec = recall_score(y_test, y_pred, average="macro")
    acc = accuracy_score(y_test, y_pred)
    print(f"  {label:<35}  macro-F1={f1:.4f}  macro-recall={rec:.4f}  accuracy={acc:.4f}")

print("=== BP test set (1141 rows) ===")
eval_on_test("tuned AdaBoost", bp_tuned,
             bp_test_Rock_Doves.drop(columns=["BP", "CHOL"]),
             np.ravel(bp_test_Rock_Doves[["BP"]]))

print("\n=== CHOL test set (1141 rows) ===")
eval_on_test("tuned RandomForest", chol_tuned,
             chol_test_Rock_Doves.drop(columns=["BP", "CHOL"]),
             np.ravel(chol_test_Rock_Doves[["CHOL"]]))

=== BP test set (1141 rows) ===
  tuned AdaBoost                       macro-F1=0.6397  macro-recall=0.6412  accuracy=0.6547

=== CHOL test set (1141 rows) ===
  tuned RandomForest                   macro-F1=0.4044  macro-recall=0.4168  accuracy=0.4549


## Model Submission

In [70]:
#Not sure what we'll need to do to make this work on the lab computer

import cloudpickle
#  Svc
SgdC.fit( X=bp_training_Rock_Doves.drop(columns=["BP", "CHOL"]), y=np.ravel(bp_training_Rock_Doves[["CHOL"]]))

# Dump your fitted model into `model.joblib` within the current directory
with open('model.pkl', 'wb') as f:
    cloudpickle.dump(SgdC, f)
from huggingface_hub import HfApi, login, create_repo

# Ensure you use a token that has 'write' access
login()
api = HfApi()
from submit import upload_files

repo_id = "Project"
upload_files(api, repo_id, display)

HfHubHTTPError: Client error '401 Unauthorized' for url 'https://huggingface.co/api/repos/create' (Request ID: Root=1-6a0001a0-26fc00de1ec3ff334a2bdb46;cad6ff43-eba2-45b4-9fe3-16ae0ae2c819)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401

Invalid username or password.